# 🎯 Train YOLOv8n Face Detection trên WIDER Face Dataset

Notebook này thực hiện:
1. Cài đặt thư viện cần thiết (ultralytics)
2. Tải model YOLOv8n pretrained
3. Tải và chuẩn bị dataset WIDER Face → format YOLO
4. Huấn luyện model (auto-save checkpoint → Drive mỗi epoch)
5. Đánh giá và xuất model

### ✨ Hỗ trợ:
- Auto-save checkpoint → Drive mỗi epoch
- Auto-resume nếu Colab ngắt
- **Chuyển sang tài khoản khác** train tiếp (upload checkpoint)

> ⚠️ Chạy trên Google Colab với GPU (T4 trở lên).

## 1. Cài đặt thư viện & Mount Drive

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

## 2. ⚙️ CẤU HÌNH — Chỉnh sửa ở đây

Nếu bạn **đổi tài khoản Gmail** để train tiếp, hãy:
1. Chọn `RESUME_MODE = 'upload'` hoặc `'url'`
2. Nếu `'upload'` → notebook sẽ cho bạn upload file `last.pt`
3. Nếu `'url'` → dán link Google Drive chia sẻ của file `last.pt`

In [ ]:
import os
import glob
import shutil

# ============================================================
# 📌 CẤU HÌNH CHÍNH — CHỈNH Ở ĐÂY
# ============================================================

# Thư mục lưu checkpoint trên Drive
DRIVE_CKPT_DIR = '/content/drive/MyDrive/yolov8_face_checkpoints'

# Cách resume khi hết GPU / đổi tài khoản:
# - 'auto'   : Tự tìm last.pt trên Drive (cùng tài khoản)
# - 'upload' : Upload file last.pt thủ công (đổi tài khoản)
# - 'url'    : Tải last.pt từ link chia sẻ Google Drive
# - 'none'   : Train từ đầu, không resume
RESUME_MODE = 'auto'

# Chỉ dùng khi RESUME_MODE = 'url'
# Paste link chia sẻ Google Drive của file last.pt ở đây:
RESUME_URL = ''
# Ví dụ: 'https://drive.google.com/file/d/1ABC.../view?usp=sharing'

# Training params
EPOCHS = 100
BATCH_SIZE = 16       # Giảm xuống 8 nếu OOM
IMG_SIZE = 640
LR0 = 0.01
LRF = 0.001
PATIENCE = 20
WORKERS = 4

# ============================================================
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
TRAIN_PROJECT = '/content/yolov8_face'
TRAIN_NAME = 'wider_face_v1'
YOLO_DIR = '/content/wider_yolo'

print(f'📁 Checkpoint Drive: {DRIVE_CKPT_DIR}')
print(f'🔄 Resume mode:     {RESUME_MODE}')
print(f'📐 Epochs={EPOCHS}, Batch={BATCH_SIZE}, ImgSize={IMG_SIZE}')

## 3. Tải model YOLOv8n pretrained

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("✅ Đã tải model YOLOv8n pretrained.")
print(model.info())

## 4. Tải dataset WIDER Face

In [ ]:
DATASET_DIR = '/content/wider_face'
os.makedirs(DATASET_DIR, exist_ok=True)

URLS = {
    'train': 'https://huggingface.co/datasets/wider_face/resolve/main/data/WIDER_train.zip',
    'val': 'https://huggingface.co/datasets/wider_face/resolve/main/data/WIDER_val.zip',
    'annot': 'https://huggingface.co/datasets/wider_face/resolve/main/data/wider_face_split.zip',
}

if not os.path.exists(f'{DATASET_DIR}/WIDER_train'):
    print('📥 Đang tải WIDER Face Train (~1.5GB)...')
    !wget -q --show-progress -O {DATASET_DIR}/WIDER_train.zip {URLS['train']}
    !unzip -q -o {DATASET_DIR}/WIDER_train.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/WIDER_train.zip
    print('✅ Done.')
else:
    print('✅ WIDER_train đã tồn tại.')

if not os.path.exists(f'{DATASET_DIR}/WIDER_val'):
    print('📥 Đang tải WIDER Face Val (~362MB)...')
    !wget -q --show-progress -O {DATASET_DIR}/WIDER_val.zip {URLS['val']}
    !unzip -q -o {DATASET_DIR}/WIDER_val.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/WIDER_val.zip
    print('✅ Done.')
else:
    print('✅ WIDER_val đã tồn tại.')

annot_found = glob.glob(f'{DATASET_DIR}/**/wider_face_train_bbx_gt.txt', recursive=True)
if not annot_found:
    print('📥 Đang tải Annotations...')
    !wget -q --show-progress -O {DATASET_DIR}/wider_face_split.zip {URLS['annot']}
    !unzip -q -o {DATASET_DIR}/wider_face_split.zip -d {DATASET_DIR}
    !rm -f {DATASET_DIR}/wider_face_split.zip
    print('✅ Done.')
else:
    print('✅ Annotations đã tồn tại.')

In [ ]:
# Auto-detect paths
print('📂 Cấu trúc dataset:')
!find {DATASET_DIR} -maxdepth 3 -type d

train_annot_list = glob.glob(f'{DATASET_DIR}/**/wider_face_train_bbx_gt.txt', recursive=True)
val_annot_list = glob.glob(f'{DATASET_DIR}/**/wider_face_val_bbx_gt.txt', recursive=True)
train_img_list = glob.glob(f'{DATASET_DIR}/**/WIDER_train/images', recursive=True)
val_img_list = glob.glob(f'{DATASET_DIR}/**/WIDER_val/images', recursive=True)

TRAIN_ANNOT_PATH = train_annot_list[0] if train_annot_list else None
VAL_ANNOT_PATH = val_annot_list[0] if val_annot_list else None
TRAIN_IMAGE_DIR = train_img_list[0] if train_img_list else None
VAL_IMAGE_DIR = val_img_list[0] if val_img_list else None

print(f'\n🔍 Auto-detected:')
print(f'  Train annot: {TRAIN_ANNOT_PATH}')
print(f'  Val annot:   {VAL_ANNOT_PATH}')
print(f'  Train imgs:  {TRAIN_IMAGE_DIR}')
print(f'  Val imgs:    {VAL_IMAGE_DIR}')

assert TRAIN_ANNOT_PATH, '❌ Không tìm thấy wider_face_train_bbx_gt.txt!'
assert VAL_ANNOT_PATH, '❌ Không tìm thấy wider_face_val_bbx_gt.txt!'
assert TRAIN_IMAGE_DIR, '❌ Không tìm thấy WIDER_train/images!'
assert VAL_IMAGE_DIR, '❌ Không tìm thấy WIDER_val/images!'
print('\n✅ OK!')

## 5. Chuyển đổi WIDER Face → YOLO format

In [ ]:
from PIL import Image

def convert_wider_to_yolo(annotation_path, image_root, output_image_dir, output_label_dir, min_face_size=10):
    """Chuyển đổi WIDER Face annotation sang YOLO format."""
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)
    stats = {'total_images': 0, 'total_faces': 0, 'skipped_faces': 0}
    
    with open(annotation_path, 'r') as f:
        while True:
            line = f.readline().strip()
            if not line:
                break
            image_rel_path = line
            image_path = os.path.join(image_root, image_rel_path)
            num_faces = int(f.readline().strip())
            
            if num_faces == 0:
                f.readline()
                continue
            if not os.path.exists(image_path):
                for _ in range(num_faces): f.readline()
                continue
            
            try:
                img = Image.open(image_path)
                img_w, img_h = img.size
                img.close()
            except:
                for _ in range(num_faces): f.readline()
                continue
            
            yolo_labels = []
            for _ in range(num_faces):
                bbox_info = f.readline().strip().split()
                x1, y1, w, h = int(bbox_info[0]), int(bbox_info[1]), int(bbox_info[2]), int(bbox_info[3])
                invalid = int(bbox_info[7]) if len(bbox_info) > 7 else 0
                if invalid == 1 or w < min_face_size or h < min_face_size:
                    stats['skipped_faces'] += 1
                    continue
                x1, y1 = max(0, x1), max(0, y1)
                w, h = min(w, img_w - x1), min(h, img_h - y1)
                if w <= 0 or h <= 0:
                    stats['skipped_faces'] += 1
                    continue
                xc = min(max((x1 + w/2) / img_w, 0), 1)
                yc = min(max((y1 + h/2) / img_h, 0), 1)
                wn = min(max(w / img_w, 0), 1)
                hn = min(max(h / img_h, 0), 1)
                yolo_labels.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
                stats['total_faces'] += 1
            
            if yolo_labels:
                flat_name = image_rel_path.replace('/', '_').replace('\\', '_')
                base_name = os.path.splitext(flat_name)[0]
                dst_image = os.path.join(output_image_dir, flat_name)
                if not os.path.exists(dst_image):
                    shutil.copy2(image_path, dst_image)
                with open(os.path.join(output_label_dir, base_name + '.txt'), 'w') as lf:
                    lf.write('\n'.join(yolo_labels) + '\n')
                stats['total_images'] += 1
    return stats

print("🔄 Đang chuyển đổi Train set...")
train_stats = convert_wider_to_yolo(TRAIN_ANNOT_PATH, TRAIN_IMAGE_DIR,
    f'{YOLO_DIR}/images/train', f'{YOLO_DIR}/labels/train')
print(f"✅ Train: {train_stats['total_images']} ảnh, {train_stats['total_faces']} faces")

print("🔄 Đang chuyển đổi Val set...")
val_stats = convert_wider_to_yolo(VAL_ANNOT_PATH, VAL_IMAGE_DIR,
    f'{YOLO_DIR}/images/val', f'{YOLO_DIR}/labels/val')
print(f"✅ Val: {val_stats['total_images']} ảnh, {val_stats['total_faces']} faces")

In [ ]:
# Kiểm tra trực quan
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import numpy as np

def visualize_yolo_sample(image_dir, label_dir, n=4):
    images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
    samples = random.sample(images, min(n, len(images)))
    fig, axes = plt.subplots(1, len(samples), figsize=(5*len(samples), 5))
    if len(samples) == 1: axes = [axes]
    for ax, name in zip(axes, samples):
        img = Image.open(os.path.join(image_dir, name))
        w, h = img.size
        ax.imshow(np.array(img))
        lbl = os.path.join(label_dir, os.path.splitext(name)[0] + '.txt')
        if os.path.exists(lbl):
            for line in open(lbl):
                p = line.strip().split()
                if len(p)==5:
                    _, xc, yc, bw, bh = map(float, p)
                    ax.add_patch(patches.Rectangle(((xc-bw/2)*w, (yc-bh/2)*h), bw*w, bh*h,
                        linewidth=2, edgecolor='lime', facecolor='none'))
        ax.set_title(name[:25], fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()

visualize_yolo_sample(f'{YOLO_DIR}/images/train', f'{YOLO_DIR}/labels/train')

## 6. Tạo dataset YAML

In [ ]:
import yaml

yaml_path = f'{YOLO_DIR}/wider_face.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({'path': YOLO_DIR, 'train': 'images/train', 'val': 'images/val',
               'nc': 1, 'names': ['face']}, f, default_flow_style=False)

print(f"✅ Dataset YAML: {yaml_path}")
print(open(yaml_path).read())

## 7. 🔄 Resume Checkpoint (đổi tài khoản)

Cell này xử lý việc lấy checkpoint để resume training.

### Cách đổi tài khoản train tiếp:
1. Trên tài khoản **cũ**: vào Drive → `yolov8_face_checkpoints/` → chuột phải `last.pt` → **Chia sẻ** → Copy link
2. Trên tài khoản **mới**: 
   - Đặt `RESUME_MODE = 'url'` và paste link vào `RESUME_URL`
   - Hoặc đặt `RESUME_MODE = 'upload'` rồi upload file `last.pt`

In [ ]:
import re

resume_checkpoint = None

def extract_gdrive_id(url):
    """Trích xuất file ID từ Google Drive URL."""
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'open\?id=([a-zA-Z0-9_-]+)',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None

if RESUME_MODE == 'auto':
    # === Tự tìm trên Drive (cùng tài khoản) ===
    drive_last = os.path.join(DRIVE_CKPT_DIR, 'last.pt')
    if os.path.exists(drive_last):
        resume_checkpoint = drive_last
        print(f'🔄 Tìm thấy checkpoint trên Drive: {drive_last}')
        print(f'   Size: {os.path.getsize(drive_last) / 1e6:.1f} MB')
    else:
        print('🆕 Không có checkpoint cũ → Train từ đầu.')

elif RESUME_MODE == 'upload':
    # === Upload thủ công (đổi tài khoản) ===
    from google.colab import files
    print('📤 Hãy upload file last.pt (từ tài khoản cũ):')
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        resume_checkpoint = f'/content/{fname}'
        # Copy vào Drive mới luôn
        shutil.copy2(resume_checkpoint, os.path.join(DRIVE_CKPT_DIR, 'last.pt'))
        print(f'✅ Đã upload: {fname} ({os.path.getsize(resume_checkpoint)/1e6:.1f} MB)')
        print(f'   Đã copy vào Drive mới: {DRIVE_CKPT_DIR}/last.pt')
    else:
        print('⚠️ Không có file nào được upload → Train từ đầu.')

elif RESUME_MODE == 'url':
    # === Tải từ link chia sẻ Google Drive ===
    if RESUME_URL:
        file_id = extract_gdrive_id(RESUME_URL)
        if file_id:
            dl_path = '/content/resume_last.pt'
            print(f'📥 Đang tải checkpoint từ Google Drive (ID: {file_id})...')
            !pip install -q gdown
            !gdown --id {file_id} -O {dl_path}
            if os.path.exists(dl_path) and os.path.getsize(dl_path) > 1000:
                resume_checkpoint = dl_path
                shutil.copy2(dl_path, os.path.join(DRIVE_CKPT_DIR, 'last.pt'))
                print(f'✅ Tải xong: {os.path.getsize(dl_path)/1e6:.1f} MB')
                print(f'   Đã copy vào Drive mới: {DRIVE_CKPT_DIR}/last.pt')
            else:
                print('❌ Tải thất bại! Kiểm tra link chia sẻ (phải public).')
        else:
            print('❌ Không thể trích xuất file ID từ URL. Kiểm tra link.')
    else:
        print('⚠️ RESUME_URL rỗng → Train từ đầu.')

elif RESUME_MODE == 'none':
    print('🆕 RESUME_MODE = none → Train từ đầu.')

# Tóm tắt
print('\n' + '=' * 50)
if resume_checkpoint:
    print(f'✅ SẼ RESUME từ: {resume_checkpoint}')
else:
    print('🆕 SẼ TRAIN TỪ ĐẦU (yolov8n.pt pretrained)')

## 8. Huấn luyện (với auto-save checkpoint mỗi epoch)

In [ ]:
from ultralytics import YOLO

# ============================
# Callbacks: Auto-save → Drive
# ============================
best_map50 = 0.0

def on_train_epoch_end(trainer):
    """Mỗi epoch: copy last.pt → Drive."""
    epoch = trainer.epoch + 1
    last_pt = trainer.save_dir / 'weights' / 'last.pt'
    if last_pt.exists():
        shutil.copy2(str(last_pt), os.path.join(DRIVE_CKPT_DIR, 'last.pt'))
        print(f'💾 [Epoch {epoch}] last.pt → Drive ✅')
    # Snapshot mỗi 10 epoch
    if epoch % 10 == 0 and last_pt.exists():
        shutil.copy2(str(last_pt), os.path.join(DRIVE_CKPT_DIR, f'epoch_{epoch}.pt'))
        print(f'💾 [Epoch {epoch}] epoch_{epoch}.pt → Drive ✅')

def on_fit_epoch_end(trainer):
    """Sau validation: lưu best model nếu cải thiện."""
    global best_map50
    epoch = trainer.epoch + 1
    best_pt = trainer.save_dir / 'weights' / 'best.pt'
    current = trainer.metrics.get('metrics/mAP50(B)', 0)
    if best_pt.exists():
        if current > best_map50:
            best_map50 = current
            shutil.copy2(str(best_pt), os.path.join(DRIVE_CKPT_DIR, 'best.pt'))
            print(f'🏆 [Epoch {epoch}] NEW BEST! mAP50={best_map50:.4f} → Drive ✅')
        else:
            shutil.copy2(str(best_pt), os.path.join(DRIVE_CKPT_DIR, 'best.pt'))

def on_train_end(trainer):
    """Khi train xong: copy final results."""
    w = trainer.save_dir / 'weights'
    for f in ['best.pt', 'last.pt']:
        src = w / f
        if src.exists():
            shutil.copy2(str(src), os.path.join(DRIVE_CKPT_DIR, f'final_{f}'))
    csv = trainer.save_dir / 'results.csv'
    if csv.exists():
        shutil.copy2(str(csv), os.path.join(DRIVE_CKPT_DIR, 'results.csv'))
    print(f'\n🎉 Training hoàn tất! Best mAP50: {best_map50:.4f}')
    print(f'📁 Checkpoint: {DRIVE_CKPT_DIR}')

# ============================
# Load model (resume hoặc train mới)
# ============================
if resume_checkpoint:
    print(f'🔄 Loading checkpoint: {resume_checkpoint}')
    model = YOLO(resume_checkpoint)
    RESUME = True
else:
    model = YOLO('yolov8n.pt')
    RESUME = False

# Đăng ký callbacks
model.add_callback('on_train_epoch_end', on_train_epoch_end)
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
model.add_callback('on_train_end', on_train_end)

# ============================
# BẮT ĐẦU TRAINING
# ============================
print(f'\n🚀 {"RESUME" if RESUME else "START"} training...')
results = model.train(
    data=f'{YOLO_DIR}/wider_face.yaml',
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    lr0=LR0,
    lrf=LRF,
    patience=PATIENCE,
    workers=WORKERS,
    device=0,
    project=TRAIN_PROJECT,
    name=TRAIN_NAME,
    exist_ok=True,
    resume=RESUME,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=1,
    plots=True,
    verbose=True,
)

## 9. Đánh giá model

In [ ]:
best_path = os.path.join(DRIVE_CKPT_DIR, 'best.pt')
if not os.path.exists(best_path):
    best_path = f'{TRAIN_PROJECT}/{TRAIN_NAME}/weights/best.pt'

best_model = YOLO(best_path)
metrics = best_model.val(data=f'{YOLO_DIR}/wider_face.yaml', imgsz=640, batch=16, device=0, plots=True)

print("\n📊 Kết quả đánh giá:")
print(f"  mAP@50:     {metrics.box.map50:.4f}")
print(f"  mAP@50-95:  {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")

In [ ]:
from IPython.display import Image as IPImage, display

result_dir = f'{TRAIN_PROJECT}/{TRAIN_NAME}'
for plot in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.png', 'P_curve.png', 'R_curve.png']:
    path = os.path.join(result_dir, plot)
    if os.path.exists(path):
        print(f"\n📈 {plot}")
        display(IPImage(filename=path, width=800))

## 10. Test inference

In [ ]:
import matplotlib.pyplot as plt
import random

best_model = YOLO(best_path)
val_images = [os.path.join(f'{YOLO_DIR}/images/val', f)
              for f in os.listdir(f'{YOLO_DIR}/images/val') if f.endswith(('.jpg','.png'))]
test_images = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flatten(), test_images):
    r = best_model.predict(img_path, conf=0.25, iou=0.45, verbose=False)
    ax.imshow(r[0].plot()[:,:,::-1])
    ax.set_title(f'{len(r[0].boxes)} faces', fontsize=10)
    ax.axis('off')
plt.suptitle('YOLOv8n Face Detection — Inference', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 11. Kiểm tra checkpoint trên Drive

In [ ]:
print(f'📁 {DRIVE_CKPT_DIR}')
print('=' * 60)
!ls -lh {DRIVE_CKPT_DIR}

print(f'\n🏆 Best mAP50: {best_map50:.4f}')
print('\n📋 Hướng dẫn đổi tài khoản train tiếp:')
print('  1. Chia sẻ file last.pt từ Drive (chuột phải → Chia sẻ → Copy link)')
print('  2. Mở notebook trên tài khoản mới')
print('  3. Cell CẤU HÌNH: đặt RESUME_MODE = "url" và paste link vào RESUME_URL')
print('  4. Hoặc: đặt RESUME_MODE = "upload" và upload file last.pt')
print('  5. Chạy tất cả cells → notebook sẽ tự resume training!')

## 12. (Tuỳ chọn) Export ONNX

In [ ]:
best_model = YOLO(best_path)
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True, dynamic=False, opset=12)
print(f"✅ ONNX: {onnx_path}")
if onnx_path and os.path.exists(onnx_path):
    shutil.copy2(onnx_path, f'{DRIVE_CKPT_DIR}/yolov8n_face_best.onnx')
    print(f"✅ ONNX → {DRIVE_CKPT_DIR}/yolov8n_face_best.onnx")

---

## 📝 Tóm tắt

### 💾 Checkpoint trên Drive:
| File | Cập nhật | Mục đích |
|------|----------|----------|
| `best.pt` | Khi mAP50 tăng | Model tốt nhất |
| `last.pt` | Mỗi epoch | Resume training |
| `epoch_N.pt` | Mỗi 10 epoch | Snapshot |
| `final_*.pt` | Khi xong | Bản cuối |

### 🔄 Đổi tài khoản train tiếp:
1. **Chia sẻ** `last.pt` từ Drive tài khoản cũ
2. Tài khoản mới: `RESUME_MODE = 'url'` + paste link
3. Hoặc: `RESUME_MODE = 'upload'` + upload file

### Tips:
- **OOM** trên T4 → giảm `BATCH_SIZE = 8`
- **Colab ngắt** → chạy lại, `RESUME_MODE = 'auto'` sẽ tự resume
- **Hết GPU** → đổi tài khoản, dùng `'url'` hoặc `'upload'`